# 01 — Forex Data Analysis: Does Recent Momentum Predict EUR/USD Returns?

**Project 001: Momentum · Lesson 01.1 · Foundations**

## Learning objectives

By the end of this lesson you will be able to:

1. Download EUR/USD 15-minute OHLC data with `yfinance` and explain its limits.
2. Clean and validate OHLC data (timestamps, duplicates, ordering, OHLC consistency).
3. Compute simple returns, log returns and candle ranges — and explain when each is appropriate.
4. Plot price, returns and ranges and read descriptive statistics.
5. Test whether the **previous 30-minute return** predicts the **subsequent 30-minute return**
   (correlation + regression + statistical significance).
6. Interpret a correlation correctly: statistical vs economic significance, and why a
   significant result is **not** yet a tradable strategy.

## Research question

> **"Does recent price momentum contain information about subsequent EUR/USD returns?"**

## How to run this notebook

- **Kaggle:** New Notebook → `!pip install -q yfinance pandas numpy matplotlib scipy statsmodels` →
  upload this `.ipynb` → Run All.
- **Local:** `pip install -r requirements.txt` → open in Jupyter Lab / VS Code.

Run cells **top to bottom**. Do not skip the interpretation cells — they are the point of the lesson.


## 2. Research paper — what the literature actually says

We anchor this lesson in **three verified papers**. They study momentum in FX, but **none of them
uses 15-minute EUR/USD spot data**. Understanding the gap is part of the lesson.

| Paper | Market & horizon | What they test | Relevance to us |
|-------|------------------|----------------|-----------------|
| **Elaut, Frömmel & Lampaert (2015)** — *Intraday momentum in FX markets* (SSRN) | RUB/USD tick data, Moscow MICEX, 2005–2014; first half-hour → last half-hour of the day | Positive intraday momentum; **not** driven by informed trading; consistent with liquidity-provider risk aversion | **Closest to us** (intraday FX, short horizon) but different pair, exchange and session structure |
| **Moskowitz, Ooi & Pedersen (2012)** — *Time series momentum*, JFE 104(2) | 58 futures markets incl. FX futures; 12-month formation, 1-month hold | Past 12-month return predicts next month; positive across asset classes | Proves time-series momentum exists — at **monthly** horizon, not 30 minutes |
| **Menkhoff et al. (2012)** — *Currency momentum strategies*, JFE 106(3) | Up to 48 currencies, monthly, 1976–2010; cross-sectional winner-minus-loser | Winner-minus-loser spread up to ~10% p.a.; distinct from carry; hard to exploit after costs | Proves **cross-sectional** FX momentum — needs many currencies, not one pair |

### Elaut et al. (2015) in detail — our primary intraday reference

- **Question:** Is there intraday momentum in FX, and is it informed trading or liquidity provision?
- **Method:** Tick-by-tick RUB/USD on MICEX (2005–2014). Regress last-half-hour return on
  first-half-hour return.
- **Finding:** Significant positive relationship (momentum). For this dataset it is **not**
  explained by informed trading; it fits risk aversion for overnight holdings among
  liquidity providers.
- **Limitations:** Single emerging-market pair, single exchange with a defined open/close
  and overnight gap, tick data. May not generalise to majors.
- **Our adaptation (explicit):** We test the *spirit* — short-horizon continuation — on
  **EUR/USD 15-min candles, 24-hour spot market, adjacent 30-min blocks**. Different pair,
  different market structure (no single open/close), different data granularity. A null
  result here does **not** refute Elaut et al., and a positive result does **not** confirm
  their mechanism.

Full citations are in Section 15. Structured notes: `research/literature_review.md`.


## 3. Research question & hypotheses

**Research question:** Does recent price momentum contain information about subsequent
EUR/USD returns?

**Operationalisation for this lesson (testable):**

- Compute **non-overlapping 30-minute returns** from 15-minute candles.
- Let `r_t` = 30-min return ending at time `t`, `r_{t+1}` = next 30-min return.
- Test whether `r_t` predicts `r_{t+1}`.

**Hypotheses:**

- **H0 (null):** `corr(r_t, r_{t+1}) = 0` — past 30-min return contains no linear
  information about the next 30-min return.
- **H1 (alternative):** `corr(r_t, r_{t+1}) ≠ 0` — there is linear predictability
  (positive = momentum/continuation, negative = reversal).

We test H0 with Pearson correlation and OLS regression. **Statistical significance alone
does not make a strategy** — costs, slippage, execution timing and out-of-sample stability
come later (Projects 004–005). This lesson only asks: *is there any signal worth investigating?*


## 4. Financial concepts

### 4.1 EUR/USD and OHLC candles

EUR/USD is the price of 1 euro in US dollars. A **15-minute candle** summarises trades in
that interval:

- **Open (O):** first price, **High (H):** highest, **Low (L):** lowest, **Close (C):** last.
- **Volume (V):** for spot FX from yfinance this is **tick volume** (number of price ticks),
  *not* traded volume. Do not use it as liquidity or for VWAP.

A **pip** for EUR/USD is 0.0001 (e.g. 1.1000 → 1.1001). Standard lot = 100,000 EUR;
pip value ≈ $10 per standard lot per pip (before conversion). We will need pip values when
we add costs in Project 004.

### 4.2 Returns

- **Simple return:** `R_t = (P_t - P_{t-1}) / P_{t-1}`. Intuitive; additive only approximately.
- **Log return:** `r_t = ln(P_t / P_{t-1})`. Additive over time; preferred for statistics.
  For small moves, `r_t ≈ R_t`.

### 4.3 Candle range

`Range = High - Low` (absolute) and `Range% = (High - Low) / Open` (relative). Range is a
crude **volatility proxy**: wide candles = larger intraday moves / uncertainty.

### 4.4 Momentum, correlation and significance

- **Momentum (time-series):** past return predicts future return with the *same sign*.
- **Pearson correlation** `ρ ∈ [-1, 1]`: linear association. `ρ > 0` → continuation,
  `ρ < 0` → reversal, `ρ ≈ 0` → no linear relationship.
- **p-value:** probability of seeing a correlation this large if H0 were true. `p < 0.05`
  is conventionally "significant" — but with thousands of candles, tiny correlations can be
  significant yet economically useless. Always report **effect size** (ρ, slope, R²) alongside p.


## 5. Mathematical formulation

Let `P_t` be the Close at 15-min bar `t`.

**Simple return (15-min):**
$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

**Log return (15-min):**
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

**Candle range:**
$$\text{Range}_t = H_t - L_t, \qquad \text{Range\%}_t = \frac{H_t - L_t}{O_t}$$

**30-minute returns (non-overlapping):** resample 15-min closes to 30-min closes
`Q_k` (every second bar), then
$$R^{(30)}_k = \frac{Q_k - Q_{k-1}}{Q_{k-1}}, \qquad r^{(30)}_k = \ln(Q_k / Q_{k-1})$$

**Momentum test (lag-1):** with `x_k = r^{(30)}_k` (previous) and `y_k = r^{(30)}_{k+1}` (subsequent):

- Pearson correlation: $\rho = \text{corr}(x, y)$
- OLS regression: $y_k = \alpha + \beta x_k + \varepsilon_k$
  - `β > 0` → momentum, `β < 0` → reversal, `β ≈ 0` → no predictability
  - Report `β`, its t-statistic, p-value, R² and 95% CI.
  - t-statistic for correlation: $t = \rho\sqrt{(n-2)/(1-\rho^2)}$

**Important:** we use **non-overlapping** 30-min blocks. Overlapping windows (e.g.
`pct_change(2)` on every 15-min bar) create artificially correlated observations and
understate standard errors — a classic source of false "significance".


## 6. Data acquisition — EUR/USD 15-minute data via yfinance

Yahoo Finance ticker for EUR/USD spot: **`EURUSD=X`**.

**Known limits (read carefully):**

- `interval="15m"` allows at most **`period="60d"`** (~60 days ≈ 5,000–6,000 candles
  including weekends gaps — FX trades nearly 24/5).
- Intraday history is **not** available for years. For longer samples we will need another
  vendor (e.g. Dukascopy, OANDA, broker feed) or resample 1-hour data — later lesson.
- `Volume` is tick volume, not traded volume.
- Data is indicative spot; bid/ask, swaps and execution costs are **not** included here.
  Costs enter in Project 004.

The cell below downloads, flattens yfinance's MultiIndex columns, and validates the frame.
If Yahoo is unreachable (common on restricted networks), it raises a clear error and offers
a **synthetic fallback** (Section 6b) so you can still test the code paths — synthetic
results must never be interpreted as market evidence.


In [ ]:
# --- 6a. Install (Kaggle) / imports ---
# On Kaggle, run this first in a separate cell if needed:
# !pip install -q yfinance pandas numpy matplotlib scipy statsmodels

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Libraries imported.")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# --- 6b. Download EUR/USD 15-minute OHLC ---
TICKER = "EURUSD=X"
PERIOD = "60d"      # maximum allowed for 15m on Yahoo
INTERVAL = "15m"

try:
    import yfinance as yf
    raw = yf.download(TICKER, period=PERIOD, interval=INTERVAL, progress=False, auto_adjust=False)
    if raw is None or len(raw) == 0:
        raise RuntimeError("yfinance returned 0 rows.")
    # yfinance >=0.2 returns MultiIndex columns (Price, Ticker); flatten to OHLCV.
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    print(f"Downloaded {len(raw)} rows for {TICKER} ({PERIOD}, {INTERVAL}).")
    print(raw.head(3))
    print(raw.tail(3))
except Exception as e:
    print(f"Download failed: {e}")
    print("Set USE_SYNTHETIC = True below to test code paths with synthetic data.")
    raw = None

In [ ]:
# --- 6c. Synthetic fallback (ONLY if download failed) ---
# ⚠️ Synthetic GBM data is for testing code paths. NEVER interpret it as market evidence.
USE_SYNTHETIC = raw is None

if USE_SYNTHETIC:
    print("⚠️ Using SYNTHETIC data — results below are NOT market evidence.")
    n = 4000
    idx = pd.date_range("2024-01-01", periods=n, freq="15min", tz="UTC")
    rng = np.random.default_rng(RANDOM_SEED)
    # Geometric Brownian motion around 1.10, tiny drift, no momentum by construction.
    log_ret = rng.normal(0.000005, 0.0006, n)
    close = 1.10 * np.exp(np.cumsum(log_ret))
    raw = pd.DataFrame({
        "Open": close,
        "High": close * (1 + np.abs(rng.normal(0, 0.0003, n))),
        "Low": close * (1 - np.abs(rng.normal(0, 0.0003, n))),
        "Close": close,
        "Volume": rng.integers(500, 5000, n),
    }, index=idx)
    print(f"Generated {len(raw)} synthetic rows.")
else:
    print("Using real yfinance data.")

## 7. Data cleaning & validation

We enforce a strict contract on every OHLC frame (the same checks live in
`src/data/loader.py::validate_ohlc`):

1. Required columns present: Open, High, Low, Close, Volume.
2. DatetimeIndex, timezone normalised to **UTC**.
3. No duplicate timestamps; strictly chronological order.
4. No NaNs in OHLC; all prices > 0.
5. OHLC consistency: `High >= max(Open, Close)` and `Low <= min(Open, Close)`.

Any violation raises an error — we **fail loudly** rather than silently trading on bad data.


In [ ]:
# --- 7a. Validation function (mirrors src/data/loader.py) ---
REQUIRED_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]

def validate_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        raise ValueError("Input DataFrame is empty.")
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    out = df.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        raise ValueError("Index must be a DatetimeIndex.")
    if out.index.tz is None:
        out.index = out.index.tz_localize("UTC")
    else:
        out.index = out.index.tz_convert("UTC")
    if out.index.duplicated().any():
        raise ValueError("Duplicate timestamps found.")
    if not out.index.is_monotonic_increasing:
        out = out.sort_index()
    ohlc = out[["Open", "High", "Low", "Close"]]
    if ohlc.isna().any().any():
        raise ValueError("NaNs found in OHLC columns.")
    if (ohlc <= 0).any().any():
        raise ValueError("Non-positive prices found in OHLC columns.")
    bad = (out["High"] < out[["Open", "Close"]].max(axis=1)) | (out["Low"] > out[["Open", "Close"]].min(axis=1))
    if bad.any():
        raise ValueError(f"OHLC inconsistency in {int(bad.sum())} rows.")
    return out

df = validate_ohlc(raw)
print(f"Validated: {len(df)} rows, {df.index.min()} → {df.index.max()}")
print(f"Timezone: {df.index.tz} | Duplicates: {int(df.index.duplicated().sum())} | NaNs (OHLC): {int(df[['Open','High','Low','Close']].isna().sum().sum())}")

In [ ]:
# --- 7b. Dataset overview ---
print(df.head())
print()
print(df.tail())
print()
print(df.dtypes)
print()
print(f"Row count: {len(df)}")
print(f"Date range: {df.index.min()} → {df.index.max()}")
# Check candle spacing (FX has weekend gaps; 15-min spacing should dominate otherwise)
diffs = df.index.to_series().diff().dropna()
print("Most common bar spacing:")
print(diffs.value_counts().head(5))

## 8. Returns & candle ranges (vectorised pandas)

All calculations are **vectorised** — no Python loops over rows. First row of any
return column is `NaN` (no previous price); we keep it and let downstream code drop NaNs
explicitly, so alignment bugs are visible.


In [ ]:
# --- 8a. Simple returns, log returns, candle ranges ---
df["simple_return"] = df["Close"].pct_change()
df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))
df["candle_range"] = df["High"] - df["Low"]
df["candle_range_pct"] = df["candle_range"] / df["Open"]
df["body_pct"] = (df["Close"] - df["Open"]).abs() / df["Open"]

print(df[["Close", "simple_return", "log_return", "candle_range", "candle_range_pct"]].head(10))
print()
print(f"NaNs in returns (expected 1): simple={int(df['simple_return'].isna().sum())}, log={int(df['log_return'].isna().sum())}")

In [ ]:
# --- 8b. Sanity checks: log ≈ simple for small moves; ranges positive ---
chk = df[["simple_return", "log_return"]].dropna()
print(f"Max |simple - log| (should be tiny): {(chk['simple_return'] - chk['log_return']).abs().max():.2e}")
print(f"Candle ranges all positive: {bool((df['candle_range'] > 0).all())}")
print(f"Mean 15-min range (price): {df['candle_range'].mean():.6f}  (~{df['candle_range'].mean()/0.0001:.1f} pips)")
print(f"Mean 15-min range (%): {df['candle_range_pct'].mean()*100:.4f}%")

## 9. Exploratory analysis — no backtest yet

This lesson is **descriptive**, not a backtest: we characterise the data before testing
any strategy. A backtest (signal → position → execution → costs → performance) starts in
Project 004. Do not judge this lesson by profitability — there is no strategy here yet.


In [ ]:
# --- 9a. Descriptive statistics: price ---
print("=== Price (Close) ===")
print(df["Close"].describe())
print()
print("=== 15-min simple returns ===")
print(df["simple_return"].describe())
print()
print("=== 15-min log returns ===")
print(df["log_return"].describe())

In [ ]:
# --- 9b. Return distribution diagnostics ---
r = df["log_return"].dropna()
print(f"n = {len(r)}")
print(f"Mean (per 15-min bar): {r.mean():.8f}")
print(f"Std (per 15-min bar):  {r.std():.8f}")
print(f"Skewness: {r.skew():.3f}  (0 = symmetric)")
print(f"Kurtosis (excess): {r.kurtosis():.3f}  (0 = normal; >0 = fat tails)")
print(f"Min / Max: {r.min():.6f} / {r.max():.6f}")
# Jarque-Bera test: H0 = returns are normally distributed
jb_stat, jb_p = stats.jarque_bera(r)
print(f"Jarque-Bera: stat={jb_stat:.1f}, p={jb_p:.3g}  (p<0.05 → reject normality; fat tails expected in FX)")
print()
print("=== Candle range ===")
print(df["candle_range"].describe())
print(df["candle_range_pct"].describe())

## 10. Momentum test — previous 30-min return vs subsequent 30-min return

**Construction (non-overlapping):** resample 15-min OHLC to 30-min OHLC
(Open=first, High=max, Low=min, Close=last, Volume=sum), then compute 30-min log returns.
`prev = r30.shift(1)`, `next = r30`. This avoids the overlapping-window bias discussed in
Section 5.

We report: Pearson `ρ` + p-value, OLS slope `β` + t-stat + p-value + R² + 95% CI, and a
scatter plot. **Effect size first, p-value second.**


In [ ]:
# --- 10a. Build non-overlapping 30-minute OHLC and returns ---
df30 = pd.DataFrame({
    "Open": df["Open"].resample("30min").first(),
    "High": df["High"].resample("30min").max(),
    "Low": df["Low"].resample("30min").min(),
    "Close": df["Close"].resample("30min").last(),
    "Volume": df["Volume"].resample("30min").sum(),
}).dropna()

df30["r30"] = np.log(df30["Close"] / df30["Close"].shift(1))
df30["prev_r30"] = df30["r30"].shift(1)
df30["next_r30"] = df30["r30"]

test = df30[["prev_r30", "next_r30"]].dropna()
print(f"30-min bars: {len(df30)} | usable (prev,next) pairs: {len(test)}")
print(test.head())
print()
print(test.describe())

In [ ]:
# --- 10b. Pearson correlation (scipy) ---
rho, p_value = stats.pearsonr(test["prev_r30"], test["next_r30"])
n = len(test)
t_stat = rho * np.sqrt((n - 2) / (1 - rho**2)) if abs(rho) < 1 else np.inf
print(f"Pearson rho (prev vs next 30-min log return): {rho:.4f}")
print(f"p-value: {p_value:.4g}")
print(f"t-statistic: {t_stat:.3f}  (df = {n-2})")
print(f"R^2 (variance explained): {rho**2:.4f}  ({rho**2*100:.2f}%)")
print()
print("Interpretation guide:")
print("  rho > 0 → momentum/continuation;  rho < 0 → reversal;  rho ≈ 0 → no linear signal.")
print("  |rho| < 0.05 is negligible even if p < 0.05 (large-n artefact).")

In [ ]:
# --- 10c. OLS regression: next = alpha + beta * prev (statsmodels) ---
X = sm.add_constant(test["prev_r30"])
model = sm.OLS(test["next_r30"], X).fit()
print(model.summary())
print()
beta = model.params["prev_r30"]
beta_p = model.pvalues["prev_r30"]
ci_low, ci_high = model.conf_int().loc["prev_r30"]
print(f"Slope beta = {beta:.4f}  (95% CI: [{ci_low:.4f}, {ci_high:.4f}], p = {beta_p:.4g})")
print(f"Intercept alpha = {model.params['const']:.6f}")
print(f"R^2 = {model.rsquared:.4f}")
print()
print("beta > 0 → momentum; beta < 0 → reversal; CI covering 0 → not significant.")

## 11. Visualizations

Four plots: (A) price, (B) 15-min log returns, (C) candle range, (D) prev-vs-next scatter
with OLS line. Read them together: a flat scatter with `rho ≈ 0` should match a noisy
returns plot — no visual "pattern" should be over-interpreted.


In [ ]:
# --- 11a. Price and returns ---
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df.index, df["Close"], linewidth=0.6, color="#1f77b4")
axes[0].set_title(f"EUR/USD 15-min Close ({TICKER}, {PERIOD})" + (" — SYNTHETIC" if USE_SYNTHETIC else ""))
axes[0].set_ylabel("Price")
axes[0].grid(alpha=0.3)

axes[1].plot(df.index, df["log_return"] * 100, linewidth=0.5, color="#d62728")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("15-min log returns (%)")
axes[1].set_ylabel("% per 15 min")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 11b. Candle range and return distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df.index, df["candle_range"] / 0.0001, linewidth=0.5, color="#2ca02c")
axes[0].set_title("Candle range (pips)")
axes[0].set_ylabel("pips")
axes[0].grid(alpha=0.3)

r = df["log_return"].dropna() * 100
axes[1].hist(r, bins=60, color="#9467bd", edgecolor="white", linewidth=0.3)
axes[1].set_title("Distribution of 15-min log returns (%)")
axes[1].set_xlabel("% per 15 min")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 11c. Prev vs next 30-min scatter + OLS line ---
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(test["prev_r30"] * 100, test["next_r30"] * 100, s=8, alpha=0.4, color="#1f77b4")
# OLS line
xs = np.linspace(test["prev_r30"].min(), test["prev_r30"].max(), 100)
ys = model.params["const"] + model.params["prev_r30"] * xs
ax.plot(xs * 100, ys * 100, color="red", linewidth=2, label=f"OLS (beta={beta:.3f})")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Previous 30-min log return (%)")
ax.set_ylabel("Subsequent 30-min log return (%)")
ax.set_title(f"Momentum test: prev vs next 30-min return (rho={rho:.3f}, p={p_value:.3g})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Research interpretation — what did we actually learn?

Answer these in order. Do not skip to "is it profitable?" — there is no strategy yet.

1. **Effect size:** What are `rho`, `beta` and R²? Is `|rho|` large enough to matter
   (e.g. `|rho| > 0.10` is already strong for 30-min FX; `|rho| < 0.03` is negligible)?
2. **Statistical significance:** What is the p-value / 95% CI? Does the CI for `beta`
   cover 0? With ~2,000–3,000 pairs, even `rho = 0.04` can be "significant" — that is a
   large-sample artefact, not an edge.
3. **Direction:** Positive (momentum) or negative (reversal)? Compare with Elaut et al.
   (positive intraday momentum in RUB/USD) — agreement or disagreement, and why the
   comparison is limited (different pair/market/horizon).
4. **Economic significance:** Even if `beta = 0.05`, a 0.10% previous move predicts only
   0.005% next — far below EUR/USD spreads (~0.5–2 pips ≈ 0.005–0.02%) plus slippage.
   Statistical ≠ economic.
5. **Robustness:** Would the result survive a different 60-day window, a different lag,
   or out-of-sample data? (Exercises test this.) One window proves nothing.

**Typical honest outcome for 30-min EUR/USD:** `rho` near zero (often |rho| < 0.05),
p-value non-significant or marginally significant, R² < 0.5%. That is a *valid finding*:
it tells us naive 30-min momentum on EUR/USD spot is weak — consistent with the idea that
major-pair intraday prices are close to a martingale at this horizon, and that published
momentum lives at longer horizons (Moskowitz et al.) or cross-sectionally (Menkhoff et al.).


In [ ]:
# --- 12a. One-page summary (run after Section 10) ---
print("=" * 60)
print("LESSON 01.1 — MOMENTUM TEST SUMMARY")
print("=" * 60)
print(f"Data: {TICKER} {INTERVAL} {PERIOD} | rows={len(df)} | {'SYNTHETIC' if USE_SYNTHETIC else 'REAL'}")
print(f"30-min pairs: n={len(test)}")
print(f"Pearson rho = {rho:.4f} (p={p_value:.4g})")
print(f"OLS beta    = {beta:.4f} (95% CI [{ci_low:.4f}, {ci_high:.4f}], p={beta_p:.4g})")
print(f"R^2         = {model.rsquared:.4f}")
print()
verdict = "MOMENTUM (positive)" if beta > 0 else "REVERSAL (negative)"
sig = "significant" if beta_p < 0.05 else "NOT significant"
print(f"Verdict: {verdict}, {sig} at 5% level.")
print("Next step: complete Exercises (Section 14) before concluding anything.")

## 13. Limitations — read before drawing conclusions

1. **Short sample:** 60 days of 15-min data is ~2 months. Any result is window-specific.
   Moskowitz et al. use decades; Menkhoff et al. use 34 years. We have neither.
2. **Single pair, single timeframe:** EUR/USD 15-min only. No cross-sectional diversification,
   no multi-timeframe check.
3. **yfinance data quality:** indicative prices, no bid/ask, tick volume only, possible
   missing bars / corporate-action-style adjustments (irrelevant for FX but the feed can
   still gap). No survivorship issue for a single major pair, but vendor risk remains.
4. **Linear test only:** Pearson/OLS detect *linear* dependence. Non-linear or
   volatility-conditioned predictability would be missed (Project 006/007).
5. **No costs:** spreads, slippage, swaps and execution timing are ignored here. A
   statistically significant `beta` can still lose money after costs (Project 004).
6. **Multiple testing:** if you try many lags/windows and report only the best, you will
   find false positives. Pre-register the test (we did: lag-1, 30-min) and report all tries.
7. **Overlapping-window trap:** we avoided it by resampling to non-overlapping 30-min bars.
   Re-running with overlapping `pct_change(2)` will inflate significance — Exercise 4.
8. **Session effects ignored:** EUR/USD behaves differently in London vs New York vs Asia.
   Pooling all hours may hide session-specific momentum (Exercise 3).


## 14. Exercises (required before Lesson 2)

Complete all six. Save your answers in a new notebook
`notebooks/01_foundations/01_exercises.ipynb` (or as a markdown file) and share the outputs
with your mentor. **Do not proceed to Lesson 2 until these are reviewed.**

### Exercise 1 — Sample sensitivity
Re-run Sections 6–10 with `PERIOD="30d"` and with `PERIOD="60d"` but dropping the first
week. Report `n`, `rho`, p-value and `beta` for each. Does the conclusion change?

### Exercise 2 — Longer lags & ACF
Compute lag-2 and lag-3 autocorrelations of 30-min log returns. Plot an autocorrelation
function (ACF) for lags 1–10 with 95% confidence bands (±1.96/√n). Is lag-1 special?

### Exercise 3 — Session split
Split 30-min pairs by hour-of-day (UTC): Asia (00–08), London (08–13), New York (13–21),
late (21–24). Compute `rho` per session. Is momentum concentrated in one session?
Relate to Elaut et al.'s liquidity-provision story.

### Exercise 4 — Overlapping-window bias
Recompute the test using overlapping 30-min returns (`Close.pct_change(2)` on 15-min data,
every bar). Compare `rho`, p-value and standard errors with the non-overlapping version.
Explain *why* overlapping understates standard errors (shared observations).

### Exercise 5 — Regression detail
Add the OLS line to the scatter (already in 11c). Report slope, intercept, R² and 95% CI
for `beta`. In one sentence: how much of next-bar variance does prev-bar explain?

### Exercise 6 — Interpretation memo (5 sentences)
Write 5 sentences covering: (1) effect size, (2) statistical significance, (3) economic
significance vs spreads, (4) comparison with Elaut et al., (5) what you would test next
and why. Distinguish statistical from economic significance explicitly.

**Completion criteria:** all six exercises answered with code + numbers + interpretation;
no claim of profitability; limitations acknowledged.


## 15. References

### Primary papers (verified — see `research/literature_review.md` for structured notes)

1. **Elaut, G., Frömmel, M., & Lampaert, K. (2015).** Intraday momentum in FX markets:
   Disentangling informed trading from liquidity provision. *SSRN Working Paper.*
   https://ssrn.com/abstract=2694985
   — Tick-by-tick RUB/USD (MICEX, 2005–2014); first half-hour → last half-hour momentum;
   attributed to liquidity-provider risk aversion, not informed trading.

2. **Moskowitz, T. J., Ooi, Y. H., & Pedersen, L. H. (2012).** Time series momentum.
   *Journal of Financial Economics*, 104(2), 228–250.
   https://doi.org/10.1016/j.jfineco.2011.11.003
   — 58 futures markets; 12-month formation predicts next month; positive across asset classes.

3. **Menkhoff, L., Sarno, L., Schmeling, M., & Schrimpf, A. (2012).** Currency momentum
   strategies. *Journal of Financial Economics*, 106(3), 660–684.
   https://doi.org/10.1016/j.jfineco.2012.06.009
   — Up to 48 currencies, monthly, 1976–2010; winner-minus-loser up to ~10% p.a.;
   distinct from carry; limits to arbitrage.

### Data & tools

4. Yahoo Finance / `yfinance` documentation — ticker `EURUSD=X`, `interval="15m"`,
   `period` max `"60d"`. https://github.com/ranaroussi/yfinance
   — Indicative spot data; FX Volume = tick volume.

### Conclusion

You now have a validated EUR/USD 15-min dataset, return/range calculations, and a
pre-registered lag-1 momentum test with effect size, significance and economic context.
Whatever `rho` you found, the lesson is complete only when you can explain *why* it is
(and is not) evidence of a tradable edge. Bring your numbers and Exercise answers to the
review — Lesson 2 builds a moving-average momentum signal on top of this foundation.
